In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objs as go
import plotly.express as px

In [2]:
# Carga de datos
simpsons = pd.read_csv("../data/raw/simpsons_episodes.csv", index_col=0)
south_park = pd.read_csv("../data/raw/SouthPark_Episodes.csv", index_col=0)
series_basics = pd.read_csv("../data/raw/series.basics.tsv.gz", sep = "\t", low_memory = False, index_col=0)
series_episodes = pd.read_csv("../data/raw/series.episode.tsv.gz", sep = "\t", low_memory = False, index_col=0)
series_ratings = pd.read_csv("../data/raw/series.ratings.tsv.gz", sep = "\t", low_memory = False, index_col=0)


In [3]:
#buscar datos de south park en el dataset original
spark_basics = series_basics[series_basics.index == "tt0121955"]
spark_eps = series_episodes[series_episodes["parentTconst"] == "tt0121955"]


In [4]:
#ajustar datos para crear el dataset limpio de south park
# unir los datos de South Park para unificarlos en uno solo
spark_data = spark_eps.join(series_ratings, how="left")
spark_tconsts = spark_data.index.tolist()
spark_titles = series_basics[series_basics.index.isin(spark_tconsts)]
spark_data = spark_data.join(spark_titles[["primaryTitle"]])

#ajustar datos de nombres y tipos en southpark considerando el dataset nuevo
spark_data = spark_data.rename(columns={
    "primaryTitle": "titulo",
    "seasonNumber": "temporada",
    "episodeNumber": "episodio"
})

spark_data["temporada"] = pd.to_numeric(
    spark_data["temporada"],
)

spark_data["episodio"] = pd.to_numeric(
    spark_data["episodio"],
)

sp_fechas = south_park[["Season", "Episode", "Air Date"]].rename(columns={
    "Season": "temporada",
    "Episode": "episodio",
    "Air Date": "fecha_estreno"
})

sp_fechas["temporada"] = pd.to_numeric(
    sp_fechas["temporada"],
)

sp_fechas["episodio"] = pd.to_numeric(
    sp_fechas["episodio"],
)

spark_data = spark_data.merge(
    sp_fechas,
    on=["temporada", "episodio"],
    how="left"
)

spark_data["fecha_estreno"] = pd.to_datetime(
    spark_data["fecha_estreno"]
)


#renombrar columnas para mejor entendimiento
spark_data = spark_data.rename(columns={
    "parentTconst": "serie_id",
    "seasonNumber": "temporada",
    "episodeNumber": "episodio",
    "averageRating": "rating",
    "numVotes": "votos"
})

#ordenar
spark_data.sort_values(by=["temporada", "episodio"], inplace=True)
spark_data = spark_data[
    ["titulo", "temporada", "episodio",
     "rating", "votos", "fecha_estreno"]
]



In [5]:
#limpiar de dataset de los simpsons para que coincida mas con el de south park
simpsons_data = simpsons[[
    "title",
    "season",
    "number_in_season",
    "imdb_rating",
    "us_viewers_in_millions",
    "original_air_date"
]].copy()

simpsons_data = simpsons_data.rename(columns={
    "title": "titulo",
    "number_in_season": "episodio",
    "season": "temporada",
    "imdb_rating": "rating",
    "original_air_date": "fecha_estreno"
})

simpsons_data.sort_values(by=["temporada", "episodio"], inplace=True)

simpsons_data["fecha_estreno"] = pd.to_datetime(
    simpsons_data["fecha_estreno"],
)


In [6]:
spark_data.to_parquet(
    "../data/processed/south_park.parquet",
    index=False
)

simpsons_data.to_parquet(
    "../data/processed/simpsons.parquet",
    index=False
)